In [ ]:
from collections import defaultdict
from DRL_algorithms.Dynamic_methods import (
    iterative_policy_evaluation_sparse,
    policy_iteration_sparse,
    value_iteration_sparse
)
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from Costum_environments.TwoRoundRPS import TwoRoundRPS

In [6]:
env = TwoRoundRPS()
ACTIONS = env.ACTIONS
S = []

# Génération de tous les états possibles
S.append(('start',))
for a in ACTIONS:
    S.append(('second', a))
S.append(('done',))

A = ACTIONS
R = [-1, 0, 1]
terminal_states = [('done',)]

# Construction du modèle
model = defaultdict(lambda: defaultdict(list))
valid_actions_dict = defaultdict(list)

# État ('start',) : choisir action → ('second', a) avec récompense vs random
for a in ACTIONS:
    for opp in ACTIONS:
        next_state = ('second', a)
        r = env._get_reward(a, opp)
        model[('start',)][a].append((1/3, next_state, r, False))
        valid_actions_dict[('start',)].append(a)

# État ('second', a1) : choisir action a2 → ('done',) avec reward vs imitation de a1
for a1 in ACTIONS:
    state = ('second', a1)
    for a2 in ACTIONS:
        opp = a1
        r = env._get_reward(a2, opp)
        model[state][a2].append((1.0, ('done',), r, True))
        valid_actions_dict[state].append(a2)


In [3]:
# Politique aléatoire
pi_random = {
    s: {a: 1 / len(valid_actions_dict[s]) for a in valid_actions_dict[s]}
    for s in S if s not in terminal_states
}
V_random = iterative_policy_evaluation_sparse(
    pi=pi_random,
    S=S,
    A=A,
    model=model,
    terminal_states=terminal_states,
    valid_actions_dict=valid_actions_dict
)

# Politique toujours 'rock'
pi_rock = {
    s: {a: 1.0 if a == 'rock' else 0.0 for a in valid_actions_dict[s]}
    for s in S if s not in terminal_states
}
V_rock = iterative_policy_evaluation_sparse(
    pi=pi_rock,
    S=S,
    A=A,
    model=model,
    terminal_states=terminal_states,
    valid_actions_dict=valid_actions_dict
)

# Politique toujours 'paper'
pi_paper = {
    s: {a: 1.0 if a == 'paper' else 0.0 for a in valid_actions_dict[s]}
    for s in S if s not in terminal_states
}
V_paper = iterative_policy_evaluation_sparse(
    pi=pi_paper,
    S=S,
    A=A,
    model=model,
    terminal_states=terminal_states,
    valid_actions_dict=valid_actions_dict
)

# Affichage des valeurs estimées
print("=== Valeurs estimées sous 3 politiques différentes ===")
for s in S:
    print(f"État {s} : Aléatoire = {V_random[s]:.3f} | Rock = {V_rock[s]:.3f} | Paper = {V_paper[s]:.3f}")


=== Valeurs estimées sous 3 politiques différentes ===
État ('start',) : Aléatoire = 0.000 | Rock = 0.000 | Paper = 0.000
État ('second', 'rock') : Aléatoire = 0.000 | Rock = 0.000 | Paper = 1.000
État ('second', 'paper') : Aléatoire = 0.000 | Rock = -1.000 | Paper = 0.000
État ('second', 'scissors') : Aléatoire = 0.000 | Rock = 1.000 | Paper = -1.000
État ('done',) : Aléatoire = 0.000 | Rock = 0.000 | Paper = 0.000


In [7]:
# Exécution de Policy Iteration
pi_opt_pi, V_opt_pi = policy_iteration_sparse(
    S=S,
    A=A,
    R=R,
    model=model,
    terminal_states=terminal_states,
    valid_actions_dict=valid_actions_dict
)

# Affichage des résultats
print("=== Politique optimale et valeurs de V (Policy Iteration) ===")
for s in S:
    if s in terminal_states:
        print(f"État {s} (terminal) → V[{s}] = {V_opt_pi[s]:.3f}")
    else:
        best_action = max(pi_opt_pi[s], key=pi_opt_pi[s].get)
        print(f"État {s} → action optimale : {best_action}, V[{s}] = {V_opt_pi[s]:.3f}")


=== Politique optimale et valeurs de V (Policy Iteration) ===
État ('start',) → action optimale : rock, V[('start',)] = 0.220
État ('second', 'rock') → action optimale : paper, V[('second', 'rock')] = 1.000
État ('second', 'paper') → action optimale : scissors, V[('second', 'paper')] = 1.000
État ('second', 'scissors') → action optimale : rock, V[('second', 'scissors')] = 0.000
État ('done',) (terminal) → V[('done',)] = 0.000


In [8]:
pi_opt_vi, V_opt_vi = value_iteration_sparse(
    S=S,
    A=A,
    R=R,
    model=model,
    terminal_states=terminal_states,
    valid_actions_dict=valid_actions_dict
)
print("=== Politique optimale et valeurs de V (Value Iteration) ===")
for s in S:
    if s in pi_opt_vi:
        best_action = max(pi_opt_vi[s], key=pi_opt_vi[s].get)
        print(f"État {s} → action optimale : {best_action}, V[{s}] = {V_opt_vi[s]:.3f}")
    else:
        print(f"État {s} (terminal) → V[{s}] = {V_opt_vi[s]:.3f}")


=== Politique optimale et valeurs de V (Value Iteration) ===
État ('start',) → action optimale : rock, V[('start',)] = 0.990
État ('second', 'rock') → action optimale : paper, V[('second', 'rock')] = 1.000
État ('second', 'paper') → action optimale : scissors, V[('second', 'paper')] = 1.000
État ('second', 'scissors') → action optimale : rock, V[('second', 'scissors')] = 1.000
État ('done',) (terminal) → V[('done',)] = 0.000
